# PCA Anomaly Detection for TLS Profiling

This notebook evaluates one-class PCA baselines on extracted TLS traffic features. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using reconstruction error as the anomaly score.


In [1]:
import sys
sys.path.append('../../src')

## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families carry the most discriminative signal for the PCA baseline.


In [2]:
FLOW = ["bs","ps", "br", "pr", "td"]
CTLS = ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"]
STLS = ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"]
REC = ["tls.rec"]

In [3]:
from pathlib import Path

import joblib
import pandas as pd

data_dir = Path("../data-ext")
#data_dir = Path("../data")

print(f"Loading extracted features from {data_dir}.")
df_train = pd.read_parquet(data_dir / "malware_train.parquet")
df_val = pd.read_parquet(data_dir / "malware_val.parquet")
df_test = pd.read_parquet(data_dir / "malware_test.parquet")
df_train_label = pd.read_parquet(data_dir / "malware_train_labels.parquet")
df_val_label = pd.read_parquet(data_dir / "malware_val_labels.parquet")
df_test_label = pd.read_parquet(data_dir / "malware_test_labels.parquet")

preprocessor_path = data_dir / "malware_preprocessor.joblib"
if not preprocessor_path.exists():
    fallback_path = data_dir / "preprocessor.joblib"
    if not fallback_path.exists():
        raise FileNotFoundError(
            f"Could not find {preprocessor_path} or {fallback_path}. Run the data preparation notebook first."
        )
    print(f"{preprocessor_path.name} not found, using {fallback_path.name} instead.")
    preprocessor_path = fallback_path

preprocessor = joblib.load(preprocessor_path)
print(f"Loaded preprocessor from {preprocessor_path}.")

Loading extracted features from ../data-ext.
Loaded preprocessor from ../data-ext/malware_preprocessor.joblib.


In [ ]:
from tls_profiling.baselines.model_pca import PCADetector, Config
from typing import Any, Dict
import numpy as np

def train_detector(train:np.ndarray) -> PCADetector:
    detector = PCADetector(Config())
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["connection_label"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["connection_label"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["connection_label"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN
    detector = train_detector(x_train_transformed)

    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)

    # evaluate on TEST
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/pca_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {"auc_score": auc, "ap_score": ap, "f1_score": f1}

def plot_curves(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="PCA PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="PCA AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label in `system`, `unknown`, and `malware`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class, used to fit the PCA detector
- `validation`: mixed traffic, used to choose the anomaly-score threshold that maximizes `F1`
- `test`: held-out mixed traffic, used only for final reporting

Feature ablation:

- each feature group is evaluated individually
- all pairwise, triple, and full combinations are also evaluated
- reported metrics are `AUC-ROC`, `AP`, and thresholded `F1`, matching the Isolation Forest notebook so the two baselines can be compared directly


In [6]:
from itertools import combinations

feature_groups = {
    "FLOW": FLOW,
    "CTLS": CTLS,
    "STLS": STLS,
    "REC": REC,
}

import pandas as pd
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in ["system", "unknown", "malware"]:
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score" : scores["f1_score"]
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.808686,0.887568,0.823535
1,FLOW,unknown,0.516428,0.885967,0.928977
2,FLOW,malware,0.329591,0.424031,0.671470
3,CTLS,system,0.935797,0.954620,0.916964
4,CTLS,unknown,0.563456,0.888691,0.928980
5,CTLS,malware,0.787502,0.792349,0.847544
6,STLS,system,0.885120,0.934045,0.914401
7,STLS,unknown,0.639500,0.912704,0.928980
8,STLS,malware,0.873770,0.827456,0.885773
9,REC,system,0.946002,0.959766,0.944149


## System Profile

The table below reports the PCA baseline for the `system` profile across all feature-group combinations. Sort by `F1Score`, `ApScore`, or `AucScore` to compare how reconstruction-error-based detection behaves relative to the Isolation Forest experiment.


In [7]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
27,"STLS, REC",system,0.964634,0.976144,0.961433
36,"FLOW, STLS, REC",system,0.965141,0.976187,0.960870
30,"FLOW, CTLS, STLS",system,0.972445,0.985174,0.947350
21,"CTLS, STLS",system,0.972635,0.984845,0.947346
42,"FLOW, CTLS, STLS, REC",system,0.972917,0.985427,0.947342
39,"CTLS, STLS, REC",system,0.972913,0.985428,0.947338
9,REC,system,0.946002,0.959766,0.944149
33,"FLOW, CTLS, REC",system,0.965333,0.980760,0.943663
24,"CTLS, REC",system,0.964878,0.980587,0.943232
18,"FLOW, REC",system,0.943061,0.956893,0.942147


## Malware Profile

This section isolates the PCA results for the `malware` profile. Comparing these rows against the Isolation Forest notebook highlights whether PCA benefits more from compact shared structure in malware traffic or struggles with intra-class diversity.


In [8]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
8,STLS,malware,0.873770,0.827456,0.885773
17,"FLOW, STLS",malware,0.878224,0.806149,0.884604
38,"FLOW, STLS, REC",malware,0.887665,0.827198,0.878624
29,"STLS, REC",malware,0.887192,0.827479,0.878191
20,"FLOW, REC",malware,0.855091,0.811131,0.860421
26,"CTLS, REC",malware,0.792842,0.692045,0.848348
5,CTLS,malware,0.787502,0.792349,0.847544
14,"FLOW, CTLS",malware,0.789272,0.689996,0.847541
35,"FLOW, CTLS, REC",malware,0.792986,0.691331,0.847341
23,"CTLS, STLS",malware,0.833028,0.751051,0.841427


## Unknown Category

The `unknown` label remains a residual class rather than a clean semantic traffic type, so these results should still be interpreted carefully. The same metric suite is kept here so the PCA baseline can be compared directly with the IF notebook under identical data splits and feature subsets.


In [9]:
unknown_df = results_df[results_df["ClassLabel"] == "unknown"].sort_values("F1Score", ascending=False)
display(unknown_df)

,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
22,"CTLS, STLS",unknown,0.694303,0.916510,0.954848
31,"FLOW, CTLS, STLS",unknown,0.693587,0.908840,0.954825
43,"FLOW, CTLS, STLS, REC",unknown,0.689737,0.907525,0.946854
40,"CTLS, STLS, REC",unknown,0.689731,0.909031,0.946759
34,"FLOW, CTLS, REC",unknown,0.637836,0.898391,0.933148
25,"CTLS, REC",unknown,0.638017,0.900092,0.933108
37,"FLOW, STLS, REC",unknown,0.671752,0.907247,0.932138
13,"FLOW, CTLS",unknown,0.547460,0.883496,0.930709
28,"STLS, REC",unknown,0.674845,0.909086,0.929348
19,"FLOW, REC",unknown,0.506103,0.869425,0.928981
